In [ ]:
from tensorflow.keras.models import load_model
from tensorflow.keras import models, layers # Ensure these are imported for model definition
import numpy as np # Ensure numpy is imported
import os # Ensure os is imported

def convolution_block(input_tensor, n_filters, kernel_size=(3, 3), batchnorm=True):
    """
    A block of Conv2D -> (BatchNormalization) -> ReLU.
    """
    # First convolutional layer
    x = layers.Conv2D(n_filters, kernel_size, padding='same', kernel_initializer='he_normal')(input_tensor)
    if batchnorm:
        x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    # Second convolutional layer
    x = layers.Conv2D(n_filters, kernel_size, padding='same', kernel_initializer='he_normal')(x)
    if batchnorm:
        x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    return x

def attention_gate(skip_connection, gating_signal, n_filters):
    """
    Attention gate mechanism.
    :param skip_connection: Feature map from the encoder path (e.g., from an earlier convolution block).
    :param gating_signal: Feature map from the deeper decoder path (after upsampling).
    :param n_filters: Number of filters for the convolutional operations within the attention gate.
    """
    # Adjusting gating signal's channels and spatial resolution to match skip connection
    g = layers.Conv2D(n_filters, (1, 1), padding='same', kernel_initializer='he_normal')(gating_signal)
    g = layers.BatchNormalization()(g)

    # Resizing g to match the spatial dimensions of skip_connection (x)
    # We need to resize g to match x's height and width.
    g = layers.Lambda(lambda t: tf.image.resize(t, size=(skip_connection.shape[1], skip_connection.shape[2]), method='bilinear'),
                      output_shape=(skip_connection.shape[1], skip_connection.shape[2], n_filters))(g)

    # Adjusting skip connection's channels
    x = layers.Conv2D(n_filters, (1, 1), padding='same', kernel_initializer='he_normal')(skip_connection)
    x = layers.BatchNormalization()(x)

    # Add the aligned feature maps
    psi = layers.add([g, x])
    psi = layers.ReLU()(psi)

    # Apply a 1x1 convolution with sigmoid activation to generate attention coefficients
    psi = layers.Conv2D(1, (1, 1), padding='same', kernel_initializer='he_normal', activation='sigmoid')(psi)

    # Multiply the skip connection with the attention coefficients
    return layers.multiply([skip_connection, psi])

def attention_unet(input_shape, n_classes, n_filters_start=16, dropout_rate=0.5, batchnorm=True):
    """
    Defines the Attention U-Net model.
    """
    inputs = layers.Input(input_shape)

    # Encoder path
    # Block 1
    conv1 = convolution_block(inputs, n_filters_start, batchnorm=batchnorm)
    pool1 = layers.MaxPooling2D((2, 2))(conv1)
    pool1 = layers.Dropout(dropout_rate * 0.5)(pool1)

    # Block 2
    conv2 = convolution_block(pool1, n_filters_start * 2, batchnorm=batchnorm)
    pool2 = layers.MaxPooling2D((2, 2))(conv2)
    pool2 = layers.Dropout(dropout_rate)(pool2)

    # Block 3
    conv3 = convolution_block(pool2, n_filters_start * 4, batchnorm=batchnorm)
    pool3 = layers.MaxPooling2D((2, 2))(conv3)
    pool3 = layers.Dropout(dropout_rate)(pool3)

    # Block 4
    conv4 = convolution_block(pool3, n_filters_start * 8, batchnorm=batchnorm)
    pool4 = layers.MaxPooling2D((2, 2))(conv4)
    pool4 = layers.Dropout(dropout_rate)(pool4)

    # Bottleneck
    bottleneck = convolution_block(pool4, n_filters_start * 16, batchnorm=batchnorm)

    # Decoder path
    # Upconv 4
    upconv4 = layers.Conv2DTranspose(n_filters_start * 8, (3, 3), strides=(2, 2), padding='same', kernel_initializer='he_normal')(bottleneck)
    # Resize upconv4 to match conv4's spatial dimensions
    upconv4 = layers.Lambda(lambda t: tf.image.resize(t, size=(conv4.shape[1], conv4.shape[2]), method='bilinear'),
                            output_shape=(conv4.shape[1], conv4.shape[2], n_filters_start * 8))(upconv4)
    attn4 = attention_gate(conv4, upconv4, n_filters_start * 8)
    merge4 = layers.concatenate([upconv4, attn4], axis=-1)
    conv_up4 = convolution_block(merge4, n_filters_start * 8, batchnorm=batchnorm)

    # Upconv 3
    upconv3 = layers.Conv2DTranspose(n_filters_start * 4, (3, 3), strides=(2, 2), padding='same', kernel_initializer='he_normal')(conv_up4)
    # Resize upconv3 to match conv3's spatial dimensions
    upconv3 = layers.Lambda(lambda t: tf.image.resize(t, size=(conv3.shape[1], conv3.shape[2]), method='bilinear'),
                            output_shape=(conv3.shape[1], conv3.shape[2], n_filters_start * 4))(upconv3)
    attn3 = attention_gate(conv3, upconv3, n_filters_start * 4)
    merge3 = layers.concatenate([upconv3, attn3], axis=-1)
    conv_up3 = convolution_block(merge3, n_filters_start * 4, batchnorm=batchnorm)

    # Upconv 2
    upconv2 = layers.Conv2DTranspose(n_filters_start * 2, (3, 3), strides=(2, 2), padding='same', kernel_initializer='he_normal')(conv_up3)
    # Resize upconv2 to match conv2's spatial dimensions
    upconv2 = layers.Lambda(lambda t: tf.image.resize(t, size=(conv2.shape[1], conv2.shape[2]), method='bilinear'),
                            output_shape=(conv2.shape[1], conv2.shape[2], n_filters_start * 2))(upconv2)
    attn2 = attention_gate(conv2, upconv2, n_filters_start * 2)
    merge2 = layers.concatenate([upconv2, attn2], axis=-1)
    conv_up2 = convolution_block(merge2, n_filters_start * 2, batchnorm=batchnorm)

    # Upconv 1
    upconv1 = layers.Conv2DTranspose(n_filters_start, (3, 3), strides=(2, 2), padding='same', kernel_initializer='he_normal')(conv_up2)
    # Resize upconv1 to match conv1's spatial dimensions
    upconv1 = layers.Lambda(lambda t: tf.image.resize(t, size=(conv1.shape[1], conv1.shape[2]), method='bilinear'),
                            output_shape=(conv1.shape[1], conv1.shape[2], n_filters_start))(upconv1)
    attn1 = attention_gate(conv1, upconv1, n_filters_start)
    merge1 = layers.concatenate([upconv1, attn1], axis=-1)
    conv_up1 = convolution_block(merge1, n_filters_start, batchnorm=batchnorm)

    # Output layer
    outputs = layers.Conv2D(n_classes, (1, 1), activation='softmax')(conv_up1)

    model = models.Model(inputs=inputs, outputs=outputs)
    return model

def sparse_dice_coef(y_true, y_pred, smooth=1e-7):
    """
    Dice coefficient for sparse (integer) labels.
    Calculates the Dice coefficient for each class and returns the mean.
    """
    # Reshape y_true to flatten it, as tf.flatten is deprecated
    y_true_f = tf.cast(tf.reshape(y_true, [-1]), tf.float32)
    # y_pred will be one-hot encoded after softmax, so we need to get the argmax
    y_pred_f = tf.cast(tf.reshape(tf.argmax(y_pred, axis=-1), [-1]), tf.float32)

    # Get the number of classes as a Python integer
    num_classes = tf.get_static_value(tf.shape(y_pred)[-1])
    if num_classes is None: # Fallback if static value isn't available, though it should be for fixed output layers
        num_classes = int(y_pred.shape[-1]) # Safely get the number of classes from shape

    dice_scores = []

    for i in range(num_classes):
        # Create one-hot versions for current class
        y_true_class = tf.cast(tf.equal(y_true_f, tf.cast(i, tf.float32)), tf.float32)
        y_pred_class = tf.cast(tf.equal(y_pred_f, tf.cast(i, tf.float32)), tf.float32)

        intersection = tf.reduce_sum(y_true_class * y_pred_class)
        union = tf.reduce_sum(y_true_class) + tf.reduce_sum(y_pred_class)
        dice = (2. * intersection + smooth) / (union + smooth)
        dice_scores.append(dice)

    return tf.reduce_mean(dice_scores)

def preprocess_data(image, mask):
    """
    Preprocesses an image and its corresponding mask.
    - Image: converts to float32 and normalizes by dividing by 255.0.
    - Mask: maps pixel values: -2 to 2 (optic cup), -1 to 1 (optic disc), 0 remains 0 (background).
            Ensures mask data type is int64.
    """
    # 1. Image preprocessing: Convert to float32 and normalize
    processed_image = image.astype(np.float32) / 255.0

    # 2. Mask preprocessing: Map values and ensure int64 type
    # Make a copy to avoid modifying the original mask array in place if it's used elsewhere
    processed_mask = mask.copy()
    # Change -2 to 2 (optic cup)
    processed_mask[processed_mask == -2] = 2
    # Change -1 to 1 (optic disc)
    processed_mask[processed_mask == -1] = 1
    # Keep 0 as 0 (background) - this is already handled if values are only -2, -1, 0

    # Ensure mask's data type is an integer type (e.g., int64)
    processed_mask = processed_mask.astype(np.int64)

    return processed_image, processed_mask

# Define the path to the best model
model_path = r"C:\Users\athet\Downloads\best_atten_unet.keras"

# Parameters for data generator (re-defined to ensure availability)
BATCH_SIZE = 8 
IMAGE_DIM = (664, 798) 
N_CHANNELS = 1 
N_CLASSES = 3 

# Load and preprocess test data with race information
all_test_images = []
all_test_masks = []
all_test_processed_images = []
all_test_processed_masks = []
all_test_races = []

print("Loading and preprocessing test data with race information...")
test_path = r"C:\Users\athet\Downloads\Problem_6_SLO_Fundus_Cup_and_Disc_Segmentation-20251117T192743Z-1-001\Problem_6_SLO_Fundus_Cup_and_Disc_Segmentation\FairSeg\Testing"

#test_path  = r"C:\Users\athet\Downloads\Problem_6_SLO_Fundus_Cup_and_Disc_Segmentation\FairSeg\Testing"
test_files = sorted(os.listdir(test_path))

for i, file_name in enumerate(test_files):
    file_path = os.path.join(test_path, file_name)
    data = np.load(file_path)

    image = data['slo_fundus']
    mask = data['disc_cup_mask']
    race = data['race']

    # Preprocess image and mask
    processed_image, processed_mask = preprocess_data(image, mask)

    # Store original image and mask
    all_test_images.append(image)
    all_test_masks.append(mask)

    # Store processed image (with channel dim) and processed mask
    all_test_processed_images.append(np.expand_dims(processed_image, axis=-1))
    all_test_processed_masks.append(processed_mask)

    # Store race information
    all_test_races.append(race)

    if (i + 1) % 100 == 0 or (i + 1) == len(test_files):
        print(f"Processed {i+1}/{len(test_files)} test files.")

print("Finished loading and preprocessing test data.")

# Convert lists to numpy arrays for easier handling
all_test_images = np.array(all_test_images)
all_test_masks = np.array(all_test_masks)
all_test_processed_images = np.array(all_test_processed_images)
all_test_processed_masks = np.array(all_test_processed_masks)
all_test_races = np.array(all_test_races)

print(f"\nShape of all_test_images: {all_test_images.shape}")
print(f"Shape of all_test_masks: {all_test_masks.shape}")
print(f"Shape of all_test_processed_images: {all_test_processed_images.shape}")
print(f"Shape of all_test_processed_masks: {all_test_processed_masks.shape}")
print(f"Shape of all_test_races: {all_test_races.shape}")

# Re-instantiate the model architecture.
input_shape = (IMAGE_DIM[0], IMAGE_DIM[1], N_CHANNELS)
reconstructed_model = attention_unet(input_shape, N_CLASSES)

# Load the weights into the newly instantiated model
reconstructed_model.load_weights(model_path)

# Compile the model with the same optimizer, loss, and metrics as during training
reconstructed_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy', sparse_dice_coef]
)

best_model = reconstructed_model # Assign to best_model for consistency

print(f"Best model weights loaded successfully into a reconstructed model from {model_path}")

print("Making predictions on the full test set...")
# Make predictions on the entire preprocessed test dataset
all_test_predictions = best_model.predict(all_test_processed_images)

print("Predictions complete. Calculating overall Dice score...")

# Calculate overall Dice score for the entire test set
# The sparse_dice_coef function expects y_true and y_pred with original shapes
# and then flattens internally. all_test_processed_masks is already in the correct format.
overall_dice = sparse_dice_coef(all_test_processed_masks, all_test_predictions).numpy()
print(f"\nOverall Test Sparse Dice Coefficient: {overall_dice:.4f}")

# Group data by race and calculate Dice score for each group
unique_races = np.unique(all_test_races)

print("\nCalculating Dice scores per racial group...")

race_names = { # Based on EDA, assuming these mappings
    0: "Unknown/Other",
    1: "White",
    2: "Black or African American",
    3: "Asian",
    # Add more mappings if other race codes exist
}

for race_code in unique_races:
    race_indices = np.where(all_test_races == race_code)[0]
    
    # Skip if no samples for this race
    if len(race_indices) == 0:
        continue
        
    # Filter true masks and predictions for the current race
    race_true_masks = all_test_processed_masks[race_indices]
    race_predictions = all_test_predictions[race_indices]
    
    # Calculate Dice score for the current race group
    race_dice = sparse_dice_coef(race_true_masks, race_predictions).numpy()
    
    race_display_name = race_names.get(race_code, f"Race {race_code}")
    print(f"  Sparse Dice Coefficient for {race_display_name}: {race_dice:.4f}")

Loading and preprocessing test data with race information...


FileNotFoundError: [WinError 3] The system cannot find the path specified: 'C:\\Users\\athet\\Downloads\\Problem_6_SLO_Fundus_Cup_and_Disc_Segmentation\\FairSeg\\Testing'